# 04 - Machine learning

Objetivo: comparar modelos, controlar overfit, escolher threshold na validacao e avaliar o teste apenas no final.


In [ ]:
# Load libraries
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score, precision_score, accuracy_score, confusion_matrix, precision_recall_curve
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
try:
    from lightgbm import LGBMClassifier
    HAS_LIGHTGBM = True
except ImportError:
    HAS_LIGHTGBM = False
try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False

RANDOM_STATE = 42
TARGET = 'Churn'
ID_COLS = ['customerID']


In [ ]:
# Prepare data and split
df = pd.read_csv('../data/raw/telco_churn_raw.csv')
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0}).astype(int)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(' ', np.nan), errors='coerce')
X = df.drop(columns=ID_COLS + [TARGET])
y = df[TARGET]
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp)


In [ ]:
# Build preprocessing inside the model pipeline
num_cols = X_train.select_dtypes(include=['number', 'bool']).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=['number', 'bool']).columns.tolist()
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
preprocess = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', encoder)]), cat_cols),
])


In [ ]:
# Compare models with cross validation and validation set
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
models = {
    'LogisticRegression': LogisticRegression(max_iter=2000, class_weight='balanced', solver='liblinear', random_state=RANDOM_STATE),
    'RandomForest_regularizada': RandomForestClassifier(n_estimators=250, max_depth=8, min_samples_leaf=25, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1),
    'GradientBoosting_sklearn': GradientBoostingClassifier(n_estimators=150, learning_rate=0.04, max_depth=2, min_samples_leaf=25, subsample=0.85, random_state=RANDOM_STATE),
    'HistGradientBoosting_regularizado': HistGradientBoostingClassifier(max_iter=150, learning_rate=0.04, max_leaf_nodes=15, l2_regularization=0.2, random_state=RANDOM_STATE),
}
if HAS_XGBOOST:
    models['XGBoost_regularizado'] = XGBClassifier(n_estimators=250, learning_rate=0.03, max_depth=3, min_child_weight=10, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.10, reg_lambda=2.00, scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1)
if HAS_LIGHTGBM:
    models['LightGBM_regularizado'] = LGBMClassifier(n_estimators=300, learning_rate=0.03, max_depth=4, num_leaves=15, min_child_samples=40, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.10, reg_lambda=2.00, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
if HAS_CATBOOST:
    models['CatBoost_regularizado'] = CatBoostClassifier(iterations=300, learning_rate=0.03, depth=4, l2_leaf_reg=6, random_seed=RANDOM_STATE, auto_class_weights='Balanced', verbose=False, allow_writing_files=False)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
rows = []
fitted_models = {}
for name, model in models.items():
    pipe = Pipeline([('preprocess', preprocess), ('model', model)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring={'roc_auc': 'roc_auc', 'pr_auc': 'average_precision'}, n_jobs=-1, return_train_score=True)
    fitted = clone(pipe).fit(X_train, y_train)
    fitted_models[name] = fitted
    train_proba = fitted.predict_proba(X_train)[:, 1]
    val_proba = fitted.predict_proba(X_val)[:, 1]
    rows.append({
        'modelo': name,
        'cv_roc_auc_media': scores['test_roc_auc'].mean(),
        'cv_pr_auc_media': scores['test_pr_auc'].mean(),
        'treino_roc_auc': roc_auc_score(y_train, train_proba),
        'validacao_roc_auc': roc_auc_score(y_val, val_proba),
        'gap_auc_treino_validacao': roc_auc_score(y_train, train_proba) - roc_auc_score(y_val, val_proba),
        'validacao_pr_auc': average_precision_score(y_val, val_proba),
    })
model_report = pd.DataFrame(rows)
model_report['score_selecao'] = model_report['validacao_roc_auc'] - model_report['gap_auc_treino_validacao'].clip(lower=0) * 0.25
model_report.sort_values('score_selecao', ascending=False)


In [ ]:
# Choose threshold on validation and evaluate test once
best_name = model_report.sort_values('score_selecao', ascending=False).iloc[0]['modelo']
best_model = fitted_models[best_name]
val_proba = best_model.predict_proba(X_val)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_val, val_proba)
threshold_table = pd.DataFrame({'threshold': np.r_[thresholds, 1.0], 'precision': precision, 'recall': recall})
threshold_table['f1'] = 2 * threshold_table['precision'] * threshold_table['recall'] / (threshold_table['precision'] + threshold_table['recall'])
candidates = threshold_table[threshold_table['recall'] >= 0.60]
best_threshold = float((candidates if len(candidates) else threshold_table).sort_values('f1', ascending=False).iloc[0]['threshold'])

test_proba = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)
final_metrics = pd.DataFrame([{
    'modelo': best_name,
    'threshold': best_threshold,
    'roc_auc': roc_auc_score(y_test, test_proba),
    'pr_auc': average_precision_score(y_test, test_proba),
    'f1': f1_score(y_test, test_pred),
    'recall': recall_score(y_test, test_pred),
    'precision': precision_score(y_test, test_pred),
    'accuracy': accuracy_score(y_test, test_pred),
}])
final_metrics
